# RES-138 ? Voyage 4 Nano vs Qwen3 Embedding 0.6B (Colab)

**This notebook orchestrates; it does not implement.** BEIR loading, MRL calibration and derivation, shard verification, exact retrieval, metrics, paired bootstrap, performance summaries, artifact writing and bundle verification all live in `dynamisrag.benchmark` and are covered by CPU-only tests.

**This notebook runs on a fresh runtime, and the approved rerun re-enters that same runtime.** A clean Colab runtime has no DynamisRAG installed, so nothing here may import it until the exact commit has been cloned, checked out detached and verified against `CODE_SHA`; the checked-out runtime and pinned model dependencies are then installed and checked against Colab's torch, CUDA and NumPy before any model metadata or weights are read. The checkout is idempotent: after the preflight is reviewed, a second `Run all` in the same runtime re-verifies the existing clone's origin and cleanliness and re-detaches it at exactly `CODE_SHA` instead of cloning again, because the approval is bound to that runtime's fingerprint.

**Run mode `preflight`:** verify the GPU/runtime, frozen sources, model prompts and pooling; calibrate native 512 against derived 512 on the frozen sample; write and print the preflight SHA; stop before corpus embeddings.

**Run mode `full`:** require an exact approved preflight for this `CODE_SHA`, `RUN_ID`, runtime fingerprint, plan and verified sources before loading a candidate for corpus work; resume verified shards; write exact retrieval, metric, bootstrap and performance evidence; verify the completed local bundle; write its manifest last; and stop.

Neither mode selects a model, sets a production default, runs TEI equivalence, measures OpenSearch index footprint or uses Docker in Colab. Those are later local evidence steps.

## 0. Parameters

In [ ]:
# --- Parameters. Paste the exact 40-character commit reported for the RES-138 harness branch.
REPO_URL = "https://github.com/Litju/DynamisRAG.git"
CODE_SHA = ""

RUN_MODE = "preflight"
APPROVED_PREFLIGHT_SHA256 = ""

DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-138"
DRIVE_RUNS_SUBDIR = "runs"
DRIVE_SOURCES_SUBDIR = "sources/beir"
DRIVE_NOTEBOOKS_SUBDIR = "notebooks"

SHARD_SIZE = 4096

CANDIDATE_DIMENSIONS = (512, 1024)

BOOTSTRAP_SEED = 138
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_CONFIDENCE = 0.95

# Frozen identities, repeated here so the notebook states what it is about to check
# without a second source of truth. The repository declares these and is authoritative;
# step 5 refuses to continue if the two ever disagree.
BEIR_SOURCES = {
    "scifact": "536e14446a0ba56ed1398ab1055f39fe852686ecad24a6306c80c490fa8e0165",
    "nfcorpus": "efe5be03f8c5b86a5870102d0599d227c8c6e2484328e68c6522560385671b0b",
    "trec-covid": "120f42a7864d2214234537733c0d2c6684e42fdfafff2c5eacf98afca6656aa0",
}

MODEL_CANDIDATES = {
    "voyageai/voyage-4-nano": "67fabc9bef010dabc5f6024aa1b1b6b93410426f",
    "Qwen/Qwen3-Embedding-0.6B": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
}

LOCAL_SCRATCH_ROOT = "/content/res138"

## 1. GPU validation

Raw `torch` and nothing else. This is the one place before the checkout that may use a
third-party package, and it may only ask `torch` a question — importing any `dynamisrag`
symbol here would fail on a runtime that has not cloned the repository yet.

Refused here rather than twenty minutes into a model load. A CPU-only runtime is not a
reduced version of this benchmark: the measured throughput on the local CPU host was
0.079–0.215 documents per second, which makes `Recall@100` identical for every candidate
and therefore useless as a discriminator.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "this Colab runtime has no usable NVIDIA GPU, and the RES-138 preflight requires one. "
        "Choose Runtime > Change runtime type > Hardware accelerator > GPU, then run this "
        "notebook from the top. A CPU-only run is not a reduced version of this benchmark: "
        "the measured throughput on the local CPU host was 0.079-0.215 documents per "
        "second, which makes Recall@100 degenerate for every candidate."
    )

print(f"CUDA available: {torch.cuda.is_available()}  devices: {torch.cuda.device_count()}")
print(f"torch {torch.__version__}  cuda {torch.version.cuda}")

## 2. Mount Drive and check the storage contract

Paths, not folder ids: the ids are recorded in the repository for provenance and
documentation, and a run that needed one in order to start would break the first time the
tree is reorganised. All active work happens under `/content/res138`; Drive holds verified
archives and finished artifacts only, and is never written a matrix byte by byte.

The paths checked here are the literals from the parameter cell. The repository's
storage contract is compared against them in step 5, once the exact commit is on the
path — before that, importing the contract module is not an option.

In [ ]:
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# Literal paths, built from the parameter cell alone.
DRIVE_RUNS_PATH = f"{DRIVE_ROOT}/{DRIVE_RUNS_SUBDIR}"
DRIVE_SOURCES_PATH = f"{DRIVE_ROOT}/{DRIVE_SOURCES_SUBDIR}"
DRIVE_NOTEBOOKS_PATH = f"{DRIVE_ROOT}/{DRIVE_NOTEBOOKS_SUBDIR}"

SCRATCH = Path(LOCAL_SCRATCH_ROOT)
REPO_DIR = SCRATCH / "repo"
HF_CACHE = SCRATCH / "hf-cache"
RUNS_ROOT = Path(DRIVE_RUNS_PATH)
BEIR_CACHE = Path(DRIVE_SOURCES_PATH)

SCRATCH.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

missing = [
    path
    for path in (DRIVE_ROOT, DRIVE_RUNS_PATH, DRIVE_SOURCES_PATH, DRIVE_NOTEBOOKS_PATH)
    if not Path(path).is_dir()
]
if missing:
    raise RuntimeError(
        f"these Drive folders do not exist: {missing}. Create them once, or check that "
        f"DRIVE_ROOT is {DRIVE_ROOT!r}."
    )

print(f"scratch     {SCRATCH}")
print(f"runs        {RUNS_ROOT}")
print(f"beir cache  {BEIR_CACHE}")
print(f"notebooks   {DRIVE_NOTEBOOKS_PATH}")

## 3. Check `CODE_SHA` locally

Shape only, with the standard library: an empty value and a branch name both fail here,
before a clone is attempted and before Drive is written to.

In [ ]:
import re

if not CODE_SHA:
    raise RuntimeError(
        "CODE_SHA is empty. Paste the exact 40-character commit reported for the RES-138 "
        "harness branch: a branch name or a tag resolves to a commit that is not known until "
        "the clone runs, so it cannot bind this run to its code."
    )
if re.fullmatch(r"[0-9a-f]{40}", CODE_SHA) is None:
    raise RuntimeError(
        f"CODE_SHA {CODE_SHA!r} is not exactly 40 lowercase hexadecimal characters. Paste the "
        "full commit; an abbreviated SHA, a branch name or a tag names a commit that is not "
        "known until the clone runs."
    )

print(f"CODE_SHA {CODE_SHA}")

## 4. Clone or re-enter the exact commit

GitHub is the only code transport — no bundle, no uploaded archive. On a fresh runtime the
repository is cloned with `--no-checkout`. When it already exists — a second `Run all` in
the same runtime, after a preflight has been approved — the existing checkout is *reused*
rather than re-cloned: it must be a directory, a Git worktree whose configured `origin` is
exactly `REPO_URL`, and clean before any revision is changed. A foreign repository, a
non-repository path or local modifications are each refused and left untouched; nothing is
deleted, reset or cleaned.

Either way the exact `CODE_SHA` is fetched, checked out detached and then *verified*:
`git rev-parse HEAD` must equal `CODE_SHA` and the tree must be clean afterwards. Colab
never commits and never pushes.

**This is where code identity is established, and it imports nothing.** The commit is
cloned or reused, checked out and verified here, but it stays off `sys.path`: the
runtime-owned components are captured next, the dependencies are installed, and only after
the installs are proven not to have moved those components is the checkout imported. The
`rev-parse` comparison above is what binds everything that follows to this exact commit.

In [ ]:
import subprocess
from pathlib import Path


def git(*arguments: str, cwd: Path | None = None) -> str:
    """Run one git command, refusing a non-zero exit rather than printing its noise."""
    completed = subprocess.run(
        ["git", *arguments],
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(arguments)} failed ({completed.returncode}): {completed.stderr.strip()[:400]}"
        )
    return completed.stdout.strip()


def require_reusable_checkout(repo: Path) -> None:
    """Refuse a path that exists and is not a clean clone of the frozen repository.

    The approved rerun re-enters this runtime, so an existing checkout is the normal
    case and must be proven reusable rather than replaced: a different origin, a
    non-repository directory or local modifications would each bind the full run to
    history that CODE_SHA does not name. Nothing here deletes, resets or cleans.
    """
    if not repo.is_dir():
        raise RuntimeError(f"{repo} exists and is not a directory. Refusing to replace it.")
    if git("rev-parse", "--is-inside-work-tree", cwd=repo) != "true":
        raise RuntimeError(f"{repo} is not a Git worktree. Refusing a foreign repository.")
    origin = git("remote", "get-url", "origin", cwd=repo)
    if origin != REPO_URL:
        raise RuntimeError(
            f"{repo} has origin {origin!r} but this run clones {REPO_URL!r}. Refusing a foreign "
            "repository: its history is not the history CODE_SHA names."
        )
    existing = git("status", "--porcelain", cwd=repo)
    if existing:
        raise RuntimeError(
            f"the existing checkout at {repo} is not clean:\n{existing[:500]}\n"
            "Refusing to change revisions over local modifications; nothing was reset or cleaned."
        )


if REPO_DIR.exists():
    require_reusable_checkout(REPO_DIR)
    print(f"reusing the verified checkout at {REPO_DIR}")
else:
    git("clone", "--no-checkout", REPO_URL, str(REPO_DIR))

git("fetch", "--depth", "1", "origin", CODE_SHA, cwd=REPO_DIR)
git("checkout", "--detach", CODE_SHA, cwd=REPO_DIR)

head = git("rev-parse", "HEAD", cwd=REPO_DIR)
if head != CODE_SHA:
    raise RuntimeError(f"checked out {head!r} but CODE_SHA is {CODE_SHA!r}")
dirty = git("status", "--porcelain", cwd=REPO_DIR)
if dirty:
    raise RuntimeError(f"the checkout is not clean:\n{dirty[:500]}")

print(f"HEAD {head}")

## 5. Capture the runtime-owned versions BEFORE the installs

Colab owns Python, torch, CUDA and NumPy; the RES-138 requirements own only the additional
model stack. The two installs that follow must therefore leave the runtime's NumPy exactly
where it is. NumPy is already imported by the live kernel before a cell runs, so a
requirements file that asked pip to replace its files would mix the old already-imported
modules with the new package and break the process.

The versions are read from installed distribution metadata rather than by importing each
library, so this capture cannot itself contribute to the problem it exists to detect.

In [ ]:
from importlib.metadata import PackageNotFoundError, version


def installed_version(name: str) -> str:
    """The installed version of one distribution, or a marker if it is absent."""
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"


TORCH_BEFORE = {
    "cuda_runtime": str(torch.version.cuda or "none"),
    "torch_cuda_build": str(torch.version.cuda or "none"),
}
NUMPY_BEFORE = installed_version("numpy")

LIBRARY_NAMES = ("sentence-transformers", "transformers", "huggingface-hub", "numpy", "torch")
LIBRARIES_BEFORE = {name: installed_version(name) for name in LIBRARY_NAMES}

for name in LIBRARY_NAMES:
    print(f"  {name}: {LIBRARIES_BEFORE[name]}")

## 6. Install the checked-out runtime dependencies

The normal project runtime dependencies come from the exact commit on disk, not from the
package index: `pip install <path>` reads only `[project.dependencies]`. PEP 735
dependency groups are not a pip concept, so the repository's `benchmark` group -- the
group that carries the local `numpy>=2.3,<2.4` constraint -- cannot be installed by this
command and does not come back in through it.

In [ ]:
runtime_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", str(REPO_DIR)],
    capture_output=True,
    text=True,
    check=False,
)
if runtime_install.returncode != 0:
    raise RuntimeError(
        f"installing the checked-out DynamisRAG runtime dependencies failed "
        f"({runtime_install.returncode}): {runtime_install.stderr[-1500:]}"
    )

print(f"installed the runtime dependencies of {REPO_DIR}")

## 7. Install the RES-138 model stack

`requirements/res138-colab.txt` adds only the four model-stack libraries the benchmark
needs, each at an exact pin, and pins no NumPy, no torch and no CUDA wheel. Voyage's
frozen repository metadata declares Transformers 4.51.3, but the same frozen repository's
custom modeling module imports Transformers APIs that 4.51.3 cannot provide. That
declaration is preserved as observed provenance; the execution stack uses Transformers
4.54.0, the minimum checked compatible runtime for this Voyage revision, together with
huggingface-hub 0.34.0. sentence-transformers 5.0.0 and tokenizers 0.21.1 remain unchanged.

Colab owns Python, torch, CUDA and NumPy; the run has already captured them, and a
requirement that pulled a replacement in would change a run after it had bound itself to
those values. The installed NumPy version is recorded later in the runtime fingerprint
rather than prescribed here.

In [ ]:
colab_install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--requirement",
        str(REPO_DIR / "requirements" / "res138-colab.txt"),
    ],
    capture_output=True,
    text=True,
    check=False,
)
if colab_install.returncode != 0:
    raise RuntimeError(
        f"installing requirements/res138-colab.txt failed ({colab_install.returncode}): "
        f"{colab_install.stderr[-1500:]}"
    )

print("installed requirements/res138-colab.txt")

## 8. Prove the installs did not move anything Colab owns

torch's installed distribution version, its CUDA runtime and NumPy are compared with the
values captured before the installs, and any drift refuses the run. torch is compared
through the installed distribution metadata rather than through `torch.__version__`,
because a pip replacement changes the files on disk while the already-imported module
keeps reporting the version it was imported as, so the live attribute cannot see the drift
this check exists to catch. This check deliberately uses the standard library and the
already-imported torch only: the repository is not on `sys.path` yet, so the decision about
whether the repository may be imported cannot depend on importing it.

In [ ]:
TORCH_AFTER = {
    "cuda_runtime": str(torch.version.cuda or "none"),
    "torch_cuda_build": str(torch.version.cuda or "none"),
}
LIBRARIES_AFTER = {name: installed_version(name) for name in LIBRARY_NAMES}
NUMPY_AFTER = LIBRARIES_AFTER["numpy"]

runtime_drift = {
    name: (before_value, after_value)
    for name, before_value, after_value in (
        ("torch", LIBRARIES_BEFORE["torch"], LIBRARIES_AFTER["torch"]),
        ("cuda_runtime", TORCH_BEFORE["cuda_runtime"], TORCH_AFTER["cuda_runtime"]),
        ("torch_cuda_build", TORCH_BEFORE["torch_cuda_build"], TORCH_AFTER["torch_cuda_build"]),
        ("numpy", NUMPY_BEFORE, NUMPY_AFTER),
    )
    if before_value != after_value
}
if runtime_drift:
    raise RuntimeError(
        f"the dependency installs changed runtime-owned components: {runtime_drift}. Colab owns "
        "torch, CUDA and NumPy; installing a replacement inside the live kernel mixes already "
        "imported modules with newly installed files and changes the numerics of a run that has "
        "already recorded these values. Start a fresh runtime and install only the pinned file."
    )

for name in LIBRARY_NAMES:
    print(f"  {name}: {LIBRARIES_BEFORE[name]} -> {LIBRARIES_AFTER[name]}")

### 8a. Verify the Voyage-compatible model stack before any Hub access

The runtime-owned drift check above must pass first. This gate then verifies the exact four
resolved model-stack versions and imports the precise Transformers API surfaces required by
Voyage's frozen custom modeling module. It performs no network request and no model-id
branching. An incompatible environment therefore fails before pinned-model metadata
verification, any model download, or model construction.

Voyage's pinned repository metadata declaring Transformers 4.51.3 remains recorded
provenance. Transformers 4.54.0 is the separate execution-stack override because it is the
minimum checked release that provides the frozen custom code's required APIs.

In [ ]:
EXPECTED_MODEL_STACK = {
    "sentence-transformers": "5.0.0",
    "transformers": "4.54.0",
    "tokenizers": "0.21.1",
    "huggingface-hub": "0.34.0",
}

model_stack_mismatch = {
    name: (expected, installed_version(name))
    for name, expected in EXPECTED_MODEL_STACK.items()
    if installed_version(name) != expected
}
if model_stack_mismatch:
    raise RuntimeError(
        "RES-138 model-stack compatibility failed before any Hugging Face model metadata "
        f"or weight load: {model_stack_mismatch}"
    )

from transformers import PreTrainedModel, Qwen3Model
from transformers.cache_utils import Cache
from transformers.masking_utils import create_causal_mask
from transformers.modeling_outputs import BaseModelOutputWithPooling
from transformers.processing_utils import Unpack
from transformers.utils import TransformersKwargs

VOYAGE_REQUIRED_TRANSFORMERS_SURFACES = (
    PreTrainedModel,
    Qwen3Model,
    Cache,
    create_causal_mask,
    BaseModelOutputWithPooling,
    Unpack,
    TransformersKwargs,
)

print("RES-138 model stack is exact and Voyage-required Transformers APIs import successfully")

## 9. Put the checkout on `sys.path` and import it

Code identity is verified, both dependency installs are complete, and the runtime-owned
components are proven unchanged. Only now is the repository imported: first `dynamisrag`
itself, then the benchmark contracts it is validated against. The printed
`dynamisrag.__file__` confirms the import resolved to the clone rather than to the copy
the runtime install also placed in site-packages.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))
import dynamisrag  # noqa: E402

print(f"dynamisrag from {dynamisrag.__file__}")
print(f"HEAD {head}")

## 10. Check the parameter cell against the contracts at this commit

The notebook repeats the frozen digests so it can state what it is about to check; the
repository is the authority. This cell makes the repetition load-bearing: if the literals
and the contracts disagree, the run stops here rather than twenty minutes later with a
digest mismatch. It is also where `Res138ColabConfig` is constructed, because the config
is validated against the checked-out code. It is the first cell that imports benchmark
code, and every dependency and runtime-owned version has been established and verified
before it runs.

In [ ]:
from dynamisrag.benchmark.contracts import (
    RES138_BEIR_SOURCES,
    RES138_BOOTSTRAP_CONFIDENCE,
    RES138_BOOTSTRAP_SAMPLES,
    RES138_BOOTSTRAP_SEED,
    RES138_CANDIDATE_DIMENSIONS,
    RES138_DRIVE_LOCATIONS,
    RES138_DRIVE_ROOT,
    RES138_MODEL_CANDIDATES,
    RES138_SHARD_SIZE,
)
from dynamisrag.benchmark.res138 import Res138ColabConfig

if RES138_DRIVE_ROOT != DRIVE_ROOT:
    raise RuntimeError(
        f"the repository declares the Drive root {RES138_DRIVE_ROOT!r} and this notebook was "
        f"given {DRIVE_ROOT!r}."
    )
if {source.workload: source.sha256 for source in RES138_BEIR_SOURCES} != BEIR_SOURCES:
    raise RuntimeError(
        "the BEIR archive digests in the parameter cell are not the frozen ones in the "
        "repository at this commit."
    )
if {entry.model_id: entry.revision for entry in RES138_MODEL_CANDIDATES} != MODEL_CANDIDATES:
    raise RuntimeError(
        "the model revisions in the parameter cell are not the frozen ones in the repository "
        "at this commit."
    )
undeclared_drive_paths = sorted(
    {DRIVE_ROOT, DRIVE_RUNS_PATH, DRIVE_SOURCES_PATH, DRIVE_NOTEBOOKS_PATH}
    - {location.path for location in RES138_DRIVE_LOCATIONS}
)
if undeclared_drive_paths:
    raise RuntimeError(
        f"these Drive paths are not in the repository storage contract: {undeclared_drive_paths}"
    )
if (RES138_SHARD_SIZE, RES138_CANDIDATE_DIMENSIONS) != (SHARD_SIZE, CANDIDATE_DIMENSIONS):
    raise RuntimeError("the shard size or the candidate dimensions are not the frozen ones.")
if (RES138_BOOTSTRAP_SEED, RES138_BOOTSTRAP_SAMPLES, RES138_BOOTSTRAP_CONFIDENCE) != (
    BOOTSTRAP_SEED,
    BOOTSTRAP_SAMPLES,
    BOOTSTRAP_CONFIDENCE,
):
    raise RuntimeError("the bootstrap policy is not the frozen one.")

config = Res138ColabConfig(
    code_sha=CODE_SHA,
    run_mode=RUN_MODE,
    approved_preflight_sha256=APPROVED_PREFLIGHT_SHA256,
    shard_size=SHARD_SIZE,
    candidate_dimensions=CANDIDATE_DIMENSIONS,
    bootstrap_seed=BOOTSTRAP_SEED,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    bootstrap_confidence=BOOTSTRAP_CONFIDENCE,
    drive_root=DRIVE_ROOT,
)

print("the parameter cell agrees with the frozen contracts at this commit")

## 11. Capture the runtime fingerprint

Python, platform, GPU name, VRAM, compute capability, NVIDIA driver, CUDA runtime, and
the resolved versions of torch, numpy, sentence-transformers, transformers and
`huggingface-hub`, hashed into `res138-runtime-v1`. The run id is *derived* from the code
commit, the GPU and that digest, so an interrupted session reopens the same folder and a
different card lands in a different one.

In [ ]:
from dynamisrag.benchmark.runner import probe_colab_runtime
from dynamisrag.benchmark.runtime import capture_runtime_fingerprint

driver = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
if driver.returncode != 0:
    raise RuntimeError(
        f"nvidia-smi could not report the driver version ({driver.returncode}): "
        f"{driver.stderr.strip()[:300]}. The runtime fingerprint requires it; it is the field "
        "that changes most often when a session reconnects onto different hardware."
    )

fingerprint = capture_runtime_fingerprint(
    probe_colab_runtime(
        code_sha=CODE_SHA, nvidia_driver_version=driver.stdout.strip().splitlines()[0]
    )
)
RUNTIME_SHA = fingerprint.sha256
RUN_ID = fingerprint.run_id

print(f"run id   {RUN_ID}")
print(f"runtime  {RUNTIME_SHA}")
for key, value in sorted(dict(fingerprint.payload).items()):
    print(f"  {key}: {value}")

## 12. Write the frozen plan and create the run directory

`run-001` is never pre-created: the folder is named by the run id and carries a manifest
binding the code commit, the runtime fingerprint, the plan digest, the generation
semantics, the shard revision and size, the candidate model revisions and the dataset
digests. Re-opening is refused unless all of them match exactly — a partially resumed run
under two identities is worse than starting again.

The dataset digests come from the frozen specs rather than from a download, because they
are frozen *before* the download; step 10 proves the archives on disk are those bytes.

In [ ]:
from dynamisrag.benchmark.contracts import RES138_BEIR_SOURCES
from dynamisrag.benchmark.res138 import (
    benchmark_plan,
    create_res138_run,
    generation_semantics_sha256,
)

plan = benchmark_plan(CODE_SHA)
PLAN_SHA = plan.sha256
plan.write(RUNS_ROOT / "res138-plan.json")

RUN_DIRECTORY, run_manifest = create_res138_run(
    runs_root=RUNS_ROOT,
    config=config,
    fingerprint=fingerprint,
    dataset_digests=[(source.workload, source.sha256) for source in RES138_BEIR_SOURCES],
)
GENERATION_SEMANTICS_SHA = generation_semantics_sha256()

print(f"plan       {PLAN_SHA}")
print(f"run        {RUN_DIRECTORY}")
print(f"identity   {run_manifest.identity_sha256}")
print(f"semantics  {GENERATION_SEMANTICS_SHA}")

## 13. Verify, cache and load the frozen BEIR archives

Download to `/content/res138`, verify the SHA-256 **before** extraction, copy the
verified archive into the Drive cache and re-verify the copy. A cached archive is
re-verified on every use: Drive is a synchronising network filesystem shared with a
browser, and the cost of the check is one hash of a file that is about to be read anyway.

In [ ]:
from dynamisrag.benchmark.res138 import verify_and_cache_beir_sources

loaded = verify_and_cache_beir_sources(
    scratch_dir=SCRATCH / "sources",
    cache_dir=BEIR_CACHE,
    extract_dir=SCRATCH / "extracted",
)

for entry in loaded:
    summary = entry.workload.summary()
    print(
        f"{summary['name']:<11} zip {entry.source.sha256[:16]}  "
        f"documents {summary['document_count']:>6}  queries {summary['query_count']:>4}  "
        f"qrels {summary['qrel_count']:>6}  excluded-no-text "
        f"{entry.report.documents_without_embedding_text}"
    )

assert {entry.workload.name: entry.source.sha256 for entry in loaded} == BEIR_SOURCES, (
    "the archives on disk are not the frozen ones"
)

## 14. Verify the pinned model metadata

At each candidate's exact revision: the model-native query and document prompts must be
the frozen ones byte for byte, the pooling mode must be the frozen one, the similarity
function must be cosine, and the model must carry its own normalisation stage. A
mismatch fails the run — nothing is substituted, because a changed prompt changes the
vectors and would invalidate the calibration and any result computed under it.

The record also carries the frozen *loading* semantics for each candidate — whether the
repository ships custom model code that must be trusted, and the compute and output dtypes
the run is bound to. Those come from the frozen candidate, not from a model card read at
execution time; if the pinned repository cannot be loaded under them, that is a preflight
failure and not something to adjust around.

In [ ]:
from dynamisrag.benchmark.res138 import verify_pinned_model_metadata
from dynamisrag.benchmark.runner import HubModelMetadataReader

model_provenance = verify_pinned_model_metadata(HubModelMetadataReader())

for entry in model_provenance:
    print(f"{entry['model_id']}@{entry['revision'][:12]}  pooling={entry['pooling_mode']}")
    print(f"  prompt pair sha256 {entry['prompt_sha256']}")
    print(
        f"  trust_remote_code={entry['trust_remote_code']}  "
        f"compute={entry['compute_dtype']}  output={entry['output_dtype']}"
    )

assert {entry["model_id"]: entry["revision"] for entry in model_provenance} == MODEL_CANDIDATES

## 15. Calibrate the Matryoshka 512 derivation — one model load per candidate

The deterministic calibration set already spans every workload, so each candidate is loaded
**once** and one pass over the set decides the shortcut separately for every
(model, path, workload): 2 models × 2 paths × 3 workloads = 12 decisions from 2 model loads.
A workload-major loop would construct six encoders, and on a metered Colab runtime that is
six downloads and six cold starts for exactly the same evidence.

Between candidates the encoder is deleted, collected and the CUDA caching allocator
emptied, so the second model does not have to share a card with the first model's dead
blocks. That cleanup lives in the GPU-only runner helper, because it is the one module
allowed to import torch; nothing about the allocator is recorded anywhere.

The gate was predeclared before this ran: worst cosine at least 0.999999, worst absolute
component difference at most 1e-5, identical calibration top-10 ordering. It is not
adjusted after seeing the numbers; a failing pair gets a native 512 pass in the full run.

In [ ]:
from dynamisrag.benchmark.contracts import RES138_MODEL_CANDIDATES as FROZEN_CANDIDATES

CALIBRATION_BATCH = 16
WORKLOADS = {entry.workload.name: entry.workload for entry in loaded}
SOURCE_DIGESTS = {entry.workload.name: entry.source.sha256 for entry in loaded}

if RUN_MODE == "preflight":
    from dynamisrag.benchmark.calibration import select_calibration_set
    from dynamisrag.benchmark.runner import calibrate_frozen_candidates

    calibration = select_calibration_set([entry.workload for entry in loaded])
    calibration_item_payloads = [item.payload() for item in calibration.items]
    candidate_runs = calibrate_frozen_candidates(
        calibration=calibration,
        candidates=FROZEN_CANDIDATES,
        batch_size=CALIBRATION_BATCH,
        cache_folder=HF_CACHE,
    )
    decisions = [decision for run in candidate_runs for decision in run.decisions]
    RUNNER_PROVENANCE = {run.candidate.model_id: dict(run.provenance) for run in candidate_runs}

    for run in candidate_runs:
        record = RUNNER_PROVENANCE[run.candidate.model_id]
        print(
            f"loaded {run.candidate.model_id}@{run.candidate.revision[:12]}  "
            f"trust_remote_code={record['trust_remote_code']}  "
            f"requested_compute={record['requested_compute_dtype']}  "
            f"observed_compute={record['observed_compute_dtype']}  "
            f"output={record['output_dtype']}"
        )

    for decision in decisions:
        suffix = f"  failed: {list(decision.failures())}" if decision.failures() else ""
        print(
            f"{decision.model_id}@{decision.model_revision[:12]} {decision.workload:<11} "
            f"{decision.kind.value:<9} "
            f"cos>={decision.minimum_cosine:.9f}  "
            f"maxdiff={decision.maximum_absolute_difference:.2e}  "
            f"top-{decision.top_k} identical={decision.identical_top_k}  "
            f"derived512_allowed={decision.derived512_allowed}{suffix}"
        )

    print(f"\ncalibration items: {len(calibration_item_payloads)}")
    print(f"decisions: {len(decisions)}")
    print(f"model loads: {len(candidate_runs)}")
else:
    from dynamisrag.benchmark.fullrun import require_full_run_approval
    from dynamisrag.benchmark.res138 import PREFLIGHT_FILENAME

    APPROVED_PREFLIGHT, decisions = require_full_run_approval(
        config=config,
        preflight_path=RUN_DIRECTORY / PREFLIGHT_FILENAME,
        fingerprint=fingerprint,
        workloads=WORKLOADS,
        source_digests=SOURCE_DIGESTS,
        plan_sha256=PLAN_SHA,
    )
    print(f"approved current preflight {APPROVED_PREFLIGHT.sha256}")
    print(f"verified MRL decisions: {len(decisions)}; no corpus embedding has started")

## 16. Write the preflight bundle

One artifact stating everything a full run would rely on: the code commit, the runtime
payload and its digest, the Drive run id, each archive's verified digest and what loading
it declared, each candidate's revision, prompt digests and loading semantics from the
pinned repository **and** what the loaded model reported about itself, the generation
semantics, the exact calibration inputs, the native-512-versus-derived-512 numbers, the
per-model-per-path-per-workload MRL decision, and the digests of the artifacts already
written. It does not authorise itself — the authorisation is a human copying its digest into
the parameter cell.

In [ ]:
from dynamisrag.benchmark.res138 import PREFLIGHT_FILENAME

preflight_path = RUN_DIRECTORY / PREFLIGHT_FILENAME

if RUN_MODE == "preflight":
    from dynamisrag.benchmark.calibration import CalibrationItem, CalibrationSet
    from dynamisrag.benchmark.res138 import merge_model_provenance, write_preflight_bundle

    model_records = merge_model_provenance(
        pinned=model_provenance, runners=list(RUNNER_PROVENANCE.values())
    )

    # Rebuild the calibration set from the payloads recorded above, so the artifact names
    # the same items the decisions were made on rather than a second selection.
    calibration_items = tuple(
        CalibrationItem(
            workload=str(payload["workload"]),
            kind=str(payload["kind"]),
            band=str(payload["band"]),
            item_id=str(payload["item_id"]),
            content_sha256=str(payload["content_sha256"]),
            length=int(payload["length"]),
            text="",
        )
        for payload in calibration_item_payloads
    )

    PREFLIGHT_SHA = write_preflight_bundle(
        preflight_path,
        config=config,
        fingerprint=fingerprint,
        run_id=RUN_ID,
        loaded=loaded,
        model_provenance=model_records,
        calibration=CalibrationSet(items=calibration_items),
        decisions=decisions,
        artifact_digests={"res138-plan.json": PLAN_SHA},
    )
    print(f"wrote {preflight_path}")
else:
    print(f"using the existing approved preflight {preflight_path}")

## 17. Print the preflight SHA-256

This is the value to review and approve. It is the digest of the artifact on disk, which
is what the full run will compare `APPROVED_PREFLIGHT_SHA256` against.

In [ ]:
from dynamisrag.benchmark.res138 import preflight_digest, verify_preflight_bundle

if RUN_MODE == "preflight":
    verify_preflight_bundle(preflight_path, expect_code_sha=CODE_SHA, expect_run_id=RUN_ID)

    print()
    print("=" * 78)
    print(f"PREFLIGHT PATH  {preflight_path}")
    print(f"PREFLIGHT SHA   {PREFLIGHT_SHA}")
    print(f"file digest     {preflight_digest(preflight_path)}")
    print(f"run id          {RUN_ID}")
    print(f"code sha        {CODE_SHA}")
    print(f"plan sha        {PLAN_SHA}")
    print("=" * 78)
    print()
    print("Review the artifact, then re-run with:")
    print('    APPROVED_PREFLIGHT_SHA256 = "' + PREFLIGHT_SHA + '"')
    print('    RUN_MODE = "full"')
else:
    print(f"current approval checked against run {RUN_ID} and runtime {RUNTIME_SHA}")

## 18. Hard stop

Nothing after this line runs in preflight mode.

In [ ]:
if RUN_MODE == "preflight":
    config.require_preflight_mode(operation="notebook_hard_stop")
    print("PREFLIGHT COMPLETE. No corpus was embedded, no metric was computed and no default was")
    print("selected. Return the preflight SHA above for review.")
    raise SystemExit("Preflight hard stop: corpus execution is not reachable in this mode.")

print("Full-run approval is exact and current. Corpus execution begins in the final cell.")

## Full run — unreachable without an approved preflight digest

Two conditions, both required: `RUN_MODE == "full"`, **and** the preflight artifact on
disk hashing to exactly `APPROVED_PREFLIGHT_SHA256`. The digest check is equality, not a
prefix: a re-written preflight — a changed driver, a different prompt, a failing MRL
decision — must not inherit an approval given to an earlier one.

This tranche implements the gate, not the pass. The cell below proves the gate and then
stops, so running this notebook cannot begin a corpus pass.

In [ ]:
if RUN_MODE != "full":
    raise SystemExit("RUN_MODE is not full; the corpus path is closed.")

from dynamisrag.benchmark.fullrun import execute_full_run
from dynamisrag.benchmark.runner import SentenceTransformersCalibrationEncoder, release_cuda_cache

report = execute_full_run(
    config=config,
    preflight_path=preflight_path,
    runs_root=RUNS_ROOT,
    scratch_root=SCRATCH,
    fingerprint=fingerprint,
    workloads=WORKLOADS,
    source_digests=SOURCE_DIGESTS,
    encoder_factory=lambda candidate: SentenceTransformersCalibrationEncoder(
        candidate=candidate, batch_size=CALIBRATION_BATCH, cache_folder=HF_CACHE
    ),
    release=release_cuda_cache,
    batch_size=CALIBRATION_BATCH,
)

print(f"FINAL RUN ID       {report.run_id}")
print(f"FULL-RUN SHA       {report.full_run_sha256}")
print(f"BUNDLE SHA         {report.bundle_sha256}")
print(f"RESULT ARTIFACTS   {len(report.result_artifacts)}")
print(f"VERIFIED FILES     {report.file_count}")
print(f"VERIFIED SHARDS    {report.shard_count}")
print(f"VERIFIED ROWS      {report.row_count}")
print("Selection not applied; TEI equivalence not run; OpenSearch index footprint not measured.")
raise SystemExit(
    "RES-138 quality/performance evidence is complete; selection and production defaults remain unapplied."
)